# Dynamixel Gripper 초기화 테스트 Notebook

C++ gripper 초기화 코드를 Python/Jupyter 스타일로 옮긴 버전입니다.

실행 순서:

1. 라이브러리 import
2. 설정값 정의
3. 포트 열기
4. Dynamixel ID ping
5. Current control mode 설정
6. Torque enable
7. Encoder/current 함수 테스트
8. Gripper 초기화 루프 실행

> ⚠️ 실제 하드웨어를 움직이는 코드입니다.  
> 처음 실행할 때는 그리퍼 주변을 비우고, 비상 정지 가능한 상태에서 실행하세요.

## 0. Import Library

In [2]:
import rby1_sdk as rby
import time
import threading
from typing import Optional, List
from dynamixel_sdk import PortHandler, PacketHandler, COMM_SUCCESS


## 1. Parameter settings

In [3]:
# Dynamixel protocol / serial 설정
PROTOCOL_VERSION = 2.0
BAUDRATE = 2_000_000                            # Default: 57600
DEVICE_NAME_GRIPPER = "/dev/rby1_gripper"

# Dynamixel control table 주소
ADDR_TORQUE_ENABLE = 64
ADDR_PRESENT_POSITION = 132
ADDR_GOAL_CURRENT = 102
ADDR_OPERATING_MODE = 11

# Control mode
CURRENT_CONTROL_MODE  = 0
POSITION_CONTROL_MODE = 3

# Torque
TORQUE_ENABLE = 1
TORQUE_DISABLE = 0

# Index
MIN_INDEX = 0
MAX_INDEX = 1

# 변환 상수
TICKS_PER_REV = 4096.0
TWO_PI = 2.0 * 3.141592
RAD2DEG = 180.0 / 3.141592

# 초기화 완료 flag
is_init_finish = False

## 2. Robot ON

In [ ]:
# ROBOT_ADDRESS = "localhost:50051"           # For Docker Simulation
ROBOT_ADDRESS = "192.168.30.1:50051"        # For Real Robot

robot = rby.create_robot_a(ROBOT_ADDRESS)   # Robot Model is "A"
model = robot.model()
if not robot.connect():
    print("Could not connect to robot")
    exit(1)
else:
    print("Successfully connected to the robot!!")
    if (robot.power_on(".*") == True):         # Turn on all the power (5v, 12v, 24v, 48v)
        time.sleep(1)
        if (robot.servo_on(".*") == True):         # Enable servo motors on all actuators
            time.sleep(1)
            robot.reset_fault_control_manager()
            robot.enable_control_manager(unlimited_mode_enabled=True)
            print("SUCCESS: Power ON the Robot!!")
            
            # 여기 추가: 그리퍼 쪽 12V 출력 ON
            robot.set_tool_flange_output_voltage("right", 12)
            robot.set_tool_flange_output_voltage("left", 12)
            print("SUCCESS: Power ON the Gripper!!")

Successfully connected to the robot!!
SUCCESS: Power ON the Robot!!
SUCCESS: Power ON the Gripper!!


In [5]:
robot.disable_control_manager()
robot.power_off(".*")  

True

## 3. PortHandler / PacketHandler 생성 및 포트 열기

In [7]:
port_handler = PortHandler(DEVICE_NAME_GRIPPER)
packet_handler = PacketHandler(PROTOCOL_VERSION)

if not port_handler.openPort():
    raise RuntimeError(f"Failed to open port: {DEVICE_NAME_GRIPPER}")

print(f"Port opened: {DEVICE_NAME_GRIPPER}")

if not port_handler.setBaudRate(BAUDRATE):
    raise RuntimeError(f"Failed to set baudrate: {BAUDRATE}")

print(f"Baudrate set: {BAUDRATE}")

SerialException: [Errno 2] could not open port /dev/rby1_gripper: [Errno 2] No such file or directory: '/dev/rby1_gripper'

## 4. Dynamixel helper 함수 정의

In [ ]:
def torque_enable(dxl_id: int, onoff: int) -> None:
    packet_handler.write1ByteTxOnly(
        port_handler,
        dxl_id,
        ADDR_TORQUE_ENABLE,
        int(onoff),
    )
    time.sleep(0.0005)


def read_operation_mode(dxl_id: int) -> Optional[int]:
    value, dxl_comm_result, dxl_error = packet_handler.read1ByteTxRx(
        port_handler,
        dxl_id,
        ADDR_OPERATING_MODE,
    )

    if dxl_comm_result == COMM_SUCCESS:
        return int(value)
    return None


def read_torque_enable(dxl_id: int) -> Optional[int]:
    value, dxl_comm_result, dxl_error = packet_handler.read1ByteTxRx(
        port_handler,
        dxl_id,
        ADDR_TORQUE_ENABLE,
    )

    if dxl_comm_result == COMM_SUCCESS:
        return int(value)
    return None


def read_encoder_rad(dxl_id: int) -> Optional[float]:
    position, dxl_comm_result, dxl_error = packet_handler.read4ByteTxRx(
        port_handler,
        dxl_id,
        ADDR_PRESENT_POSITION,
    )

    if dxl_comm_result == COMM_SUCCESS:
        # C++ 코드와 동일한 변환:
        # position / 4096 * 2pi
        return float(position) / TICKS_PER_REV * TWO_PI

    return None


def send_operation_mode(dxl_id: int, operation_mode: int) -> None:
    packet_handler.write1ByteTxOnly(
        port_handler,
        dxl_id,
        ADDR_OPERATING_MODE,
        int(operation_mode),
    )
    time.sleep(0.0005)


def send_current_ampere(dxl_id: int, current_ampere: float) -> None:
    # C++ 코드:
    # int32_t current_value = (int)(current / 2.69 * 1000.)
    current_value = int(current_ampere / 2.69 * 1000.0)

    packet_handler.write2ByteTxOnly(
        port_handler,
        dxl_id,
        ADDR_GOAL_CURRENT,
        current_value,
    )

## 5. ID 0, 1 ping 테스트

In [ ]:
active_ids_gripper: List[int] = []

for dxl_id in range(2):
    model_number, dxl_comm_result, dxl_error = packet_handler.ping(
        port_handler,
        dxl_id,
    )

    if dxl_comm_result == COMM_SUCCESS:
        print(f"Dynamixel ID {dxl_id} is active. model_number={model_number}")
        active_ids_gripper.append(dxl_id)
    else:
        print(f"Dynamixel ID {dxl_id} is NOT active.")

print("active_ids_gripper:", active_ids_gripper)

if len(active_ids_gripper) != 2:
    raise RuntimeError("Unable to ping all devices for grippers")

## 6. Current control mode 설정 + torque enable

In [ ]:
for dxl_id in active_ids_gripper:
    # Operating mode 변경 전 torque off 권장
    torque_enable(dxl_id, TORQUE_DISABLE)

    send_operation_mode(dxl_id, CURRENT_CONTROL_MODE)
    print(f"ID {dxl_id}: set current control mode")

    torque_enable(dxl_id, TORQUE_ENABLE)
    print(f"ID {dxl_id}: torque enabled")

    mode = read_operation_mode(dxl_id)
    torque = read_torque_enable(dxl_id)

    print(f"ID {dxl_id}: mode={mode}, torque={torque}")

## 7. Encoder 읽기 테스트

In [ ]:
for dxl_id in active_ids_gripper:
    q_rad = read_encoder_rad(dxl_id)

    if q_rad is None:
        print(f"ID {dxl_id}: encoder read failed")
    else:
        print(f"ID {dxl_id}: q = {q_rad:.6f} rad, {q_rad * RAD2DEG:.2f} deg")

## 8. 전류 명령 테스트: 0A 보내기

In [ ]:
for dxl_id in active_ids_gripper:
    send_current_ampere(dxl_id, 0.0)
    print(f"ID {dxl_id}: current command = 0.0 A")

## 9. Gripper 초기화 함수

C++의 `control_loop_for_gripper()`를 Python으로 옮긴 함수입니다.

동작 방식:

- ID 0, 1 각각 current control mode인지 확인
- torque enable인지 확인
- `+0.5A`, `-0.5A`를 3초 간격으로 번갈아 인가
- 각 방향에서 encoder 값을 저장
- min/max 차이가 약 `540도 * 0.9` 이상이면 초기화 완료
- 마지막에 current 0A 전송

In [ ]:
def control_loop_for_gripper(
    active_ids: List[int],
    init_current_ampere: float = 0.5,
    required_range_deg: float = 540.0 * 0.9,
    sleep_sec: float = 3.0,
    max_iterations: int = 20,
):
    global is_init_finish

    if len(active_ids) != 2:
        raise RuntimeError(
            "The number of Dynamixels for hand gripper does not match the configuration"
        )

    # C++ 코드의 q_min_max_vector와 같은 역할
    # key: dxl_id, value: [min_candidate, max_candidate]
    q_min_max = {
        dxl_id: [0.0, 0.0]
        for dxl_id in active_ids
    }

    cnt = 0

    while cnt < max_iterations:
        is_init = True

        print("=" * 60)
        print(f"Iteration {cnt}")

        for dxl_id in active_ids:
            # 1) current control mode 보장
            while True:
                operation_mode = read_operation_mode(dxl_id)

                if operation_mode is not None:
                    if operation_mode != CURRENT_CONTROL_MODE:
                        torque_enable(dxl_id, TORQUE_DISABLE)
                        send_operation_mode(dxl_id, CURRENT_CONTROL_MODE)
                        torque_enable(dxl_id, TORQUE_ENABLE)
                        print(f"ID {dxl_id}: try to change control mode")
                    else:
                        break

                time.sleep(0.1)

            # 2) torque enable 보장
            while True:
                torque = read_torque_enable(dxl_id)

                if torque is not None:
                    if not torque:
                        torque_enable(dxl_id, TORQUE_ENABLE)
                        print(f"ID {dxl_id}: try to enable torque")
                    else:
                        break

                time.sleep(0.1)

            # 3) encoder 읽고 +전류 / -전류 번갈아 인가
            q_rad = read_encoder_rad(dxl_id)

            if q_rad is not None:
                q_min_max[dxl_id][cnt % 2] = q_rad

            if cnt % 2 == 0:
                send_current_ampere(dxl_id, +init_current_ampere)
                direction = "+"
            else:
                send_current_ampere(dxl_id, -init_current_ampere)
                direction = "-"

            # 4) range 확인
            q_range_deg = abs(
                q_min_max[dxl_id][MAX_INDEX] - q_min_max[dxl_id][MIN_INDEX]
            ) * RAD2DEG

            if q_range_deg < required_range_deg:
                is_init = False

            print(f"ID {dxl_id}")
            print(f"  current direction: {direction}{init_current_ampere} A")
            print(f"  q[MIN_INDEX]: {q_min_max[dxl_id][MIN_INDEX] * RAD2DEG:.2f} deg")
            print(f"  q[MAX_INDEX]: {q_min_max[dxl_id][MAX_INDEX] * RAD2DEG:.2f} deg")
            print(f"  range: {q_range_deg:.2f} deg")
            print(f"  is_init so far: {is_init}")

        # 5) 초기화 완료 처리
        if is_init:
            print("Initialization condition satisfied.")

            for dxl_id in active_ids:
                if q_min_max[dxl_id][MIN_INDEX] > q_min_max[dxl_id][MAX_INDEX]:
                    q_min_max[dxl_id][MIN_INDEX], q_min_max[dxl_id][MAX_INDEX] = (
                        q_min_max[dxl_id][MAX_INDEX],
                        q_min_max[dxl_id][MIN_INDEX],
                    )

                send_current_ampere(dxl_id, 0.0)

            time.sleep(3.0)
            is_init_finish = True

            print("OK")
            return q_min_max

        cnt += 1
        time.sleep(sleep_sec)

    # 안전상 max_iterations 이후 정지
    for dxl_id in active_ids:
        send_current_ampere(dxl_id, 0.0)

    raise TimeoutError(
        f"Gripper initialization failed within {max_iterations} iterations"
    )

## 10. Gripper 초기화 실행

In [ ]:
q_min_max_result = control_loop_for_gripper(
    active_ids_gripper,
    init_current_ampere=0.5,
    required_range_deg=540.0 * 0.9,
    sleep_sec=3.0,
    max_iterations=20,
)

q_min_max_result

## 11. 결과를 degree 단위로 확인

In [ ]:
for dxl_id, q_pair in q_min_max_result.items():
    q_min_deg = q_pair[MIN_INDEX] * RAD2DEG
    q_max_deg = q_pair[MAX_INDEX] * RAD2DEG
    q_range_deg = abs(q_max_deg - q_min_deg)

    print(f"ID {dxl_id}")
    print(f"  min:   {q_min_deg:.2f} deg")
    print(f"  max:   {q_max_deg:.2f} deg")
    print(f"  range: {q_range_deg:.2f} deg")

## 12. 정지 및 포트 닫기

In [ ]:
# 항상 마지막에 current 0A + torque off + port close
for dxl_id in active_ids_gripper:
    try:
        send_current_ampere(dxl_id, 0.0)
        torque_enable(dxl_id, TORQUE_DISABLE)
        print(f"ID {dxl_id}: current 0A, torque disabled")
    except Exception as e:
        print(f"ID {dxl_id}: failed to stop safely:", e)

try:
    port_handler.closePort()
    print("Port closed.")
except Exception as e:
    print("Failed to close port:", e)